
# Late Delivery Prediction — Machine Learning

## Project Overview

This project develops a **binary classification model** to predict whether an e-commerce order will arrive late using the Brazilian Olist e-commerce dataset.

The goal is to identify orders at risk of late delivery using information available **at the time of purchase**, helping businesses improve delivery reliability and customer satisfaction.

### Objectives
- Prepare and explore historical order data from PostgreSQL.
- Engineer predictive features while preventing data leakage.
- Train and compare Logistic Regression and Random Forest classifiers.
- Evaluate model performance using precision, recall, F1-score, and confusion matrices.

### Prediction Target
- **0 — Not Late:** Delivered on or before the estimated delivery date.
- **1 — Late:** Delivered after the estimated delivery date.

Orders with missing delivery outcomes will be excluded from training.


## 1. Data Extraction

Load historical order data from the PostgreSQL `retail_bi` database using Pandas and SQLAlchemy.

We'll begin with order-level information and our binary prediction target (`is_late`). Additional features will be introduced during feature engineering.

In [ ]:
# Install the PostgreSQL driver into the notebook's Python environment, psycopg2 handles PostgreSQL communication
# %pip install psycopg2-binary

import os
import pandas as pd
# SQLAlchemy manages database connections
from sqlalchemy import create_engine, text
from getpass import getpass

# Prompt for the database password without displaying it in the notebook
db_password = getpass("PostgreSQL password: ")

# Create a connection to our local PostgreSQL database
engine = create_engine(
    "postgresql+psycopg2://",
    connect_args={
        "host": "localhost",
        "port": 5432,
        "dbname": "retail_bi",
        "user": "postgres",
        "password": db_password
    }
)

# Execute a simple SQL query to verify the database connection
with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Connection successful:", result.scalar())

Connection successful: 1


In [ ]:
# SQL query to retrieve historical orders and create our prediction target
query = """
SELECT
    order_id,
    customer_id,
    order_purchase_timestamp,
    order_estimated_delivery_date,
    delivery_status,

    -- Binary classification target: 1 = Late, 0 = Not Late
    CASE
        WHEN delivery_status = 'Late' THEN 1
        ELSE 0
    END AS is_late

FROM orders

-- Exclude orders with missing delivery outcomes
WHERE delivery_status IN ('Early', 'On Time', 'Late')
"""

# Execute the SQL query and load the results into a Pandas DataFrame
df = pd.read_sql(query, engine)

# Preview the first five rows
print(df.head())

# Display the number of rows and columns in our dataset
print("Dataset shape:", df.shape)

# Count how many orders belong to each target class
print("\nTarget distribution:")
print(df["is_late"].value_counts())

Dataset shape: (96476, 6)

Target distribution:
is_late
0    89941
1     6535
Name: count, dtype: int64


In [ ]:
# Close all idle database connections managed by the engine
engine.dispose()